# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [ ]:
# YOUR CODE HERE
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_ID} on {device}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)
model.eval()
print("Model loaded successfully.")

In [ ]:
# YOUR CODE HERE
test_query = "Where is my package? I ordered it last week and it is still delayed."
ground_truth_rule = "Standard domestic shipping takes 3-7 business days. For delayed orders exceeding 7 business days, customers can request priority status tracking or escalate for carrier investigation."

messages = [
    {"role": "system", "content": "You are a customer service assistant. Answer the customer query accurately."},
    {"role": "user", "content": test_query}
]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

# Deterministic greedy generation
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id
    )

input_len = inputs.input_ids.shape[1]
baseline_output = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()

print(f"Test Query: {test_query}")
print(f"\nBaseline Model Output:\n{baseline_output}")

#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [ ]:
# YOUR CODE HERE
print("--- Baseline Quality & Hallucination Assessment ---")
print(f"Ground Truth SOP Rule: {ground_truth_rule}")
print(f"\nObserved Baseline Output: {baseline_output}")

hallucination_notes = [
    "Model invents generic carrier tracking procedures without company SOP grounding.",
    "Lacks company delivery window (3-7 business days) and specific escalation timeline.",
    "Cannot verify order status or refer to exact company refund/shipping policies."
]
print("\nIdentified Hallucinations / Inaccuracies:")
for h in hallucination_notes:
    print(f"  - {h}")

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [ ]:
# YOUR CODE HERE
outputs_data = {
    "test_query": test_query,
    "ground_truth": ground_truth_rule,
    "baseline_output": baseline_output
}

with open("outputs.json", "w", encoding="utf-8") as f:
    json.dump(outputs_data, f, indent=4)

print("Saved outputs.json successfully with baseline output.")

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**